# Elite Coffee Analytics Notebook
Daniel Mutembesa | AI + Field Systems + RCT-ready

This notebook demonstrates production-grade analytics for SurveyCTO data aligned with World Bank DECPL requirements.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

df = pd.read_csv('coffee_baseline.csv')
df = df.dropna(subset=['farmer_id']).drop_duplicates('farmer_id')
df.head()

In [ ]:
# Feature Engineering
df['uses_mulch_bin'] = df['uses_mulch'].str.lower().map({'yes':1,'no':0})
df['uses_shade_bin'] = df['uses_shade'].str.lower().map({'yes':1,'no':0})
df['csa_index'] = df['uses_mulch_bin'] + df['uses_shade_bin']
df['yield_per_tree'] = df['last_yield'] / df['coffee_trees']
df.loc[df['coffee_trees']<=0,'yield_per_tree']=np.nan


In [ ]:
# Data Quality Dashboard
missing = df.isnull().mean().sort_values(ascending=False)
missing.plot(kind='bar')
plt.title('Missing Data % by Variable')
plt.show()

In [ ]:
# CSA Adoption vs Yield
sns.boxplot(x='csa_index', y='yield_per_tree', data=df)
plt.title('CSA Adoption vs Yield')
plt.show()

In [ ]:
# RCT Simulation
df['treatment'] = (df['district'].str.lower()=='wakiso').astype(int)
treat = df[df['treatment']==1]['yield_per_tree'].dropna()
control = df[df['treatment']==0]['yield_per_tree'].dropna()
print('Treatment mean:', treat.mean())
print('Control mean:', control.mean())

In [ ]:
# Simple T-test
from scipy.stats import ttest_ind
t_stat, p_val = ttest_ind(treat, control, equal_var=False)
print('T-stat:', t_stat)
print('P-value:', p_val)

In [ ]:
# Regression (CSA impact)
df_reg = df.dropna(subset=['yield_per_tree','csa_index'])
X = sm.add_constant(df_reg[['csa_index']])
y = df_reg['yield_per_tree']
model = sm.OLS(y,X).fit()
print(model.summary())

In [ ]:
# Mock NDVI integration (AI layer)
df['ndvi'] = np.random.uniform(0.3,0.9,len(df))
sns.scatterplot(x='ndvi', y='yield_per_tree', data=df)
plt.title('NDVI vs Yield (Simulated)')
plt.show()

In [ ]:
# Auto Summary Report
summary = {
 'avg_yield': df['yield_per_tree'].mean(),
 'csa_adoption_rate': df['csa_index'].mean(),
 'treatment_effect': treat.mean() - control.mean()
}
print(summary)